In [5]:
#Load Data set
DATASET_PATH =  r"C:\Users\94772\Desktop\Abdullah_project\Tamil_Character_Recognition\dataset_raw"

IMG_SIZE = (224,224)
BATCH_SIZE = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.3,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.3,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = val_test_ds.take(int(0.5 * len(val_test_ds)))
test_ds = val_test_ds.skip(int(0.5 * len(val_test_ds)))

NUM_CLASSES = len(train_ds.class_names)
print("Classes =", NUM_CLASSES)


Found 24700 files belonging to 247 classes.
Using 17290 files for training.
Found 24700 files belonging to 247 classes.
Using 7410 files for validation.
Classes = 247


In [7]:
#prefetch
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)


In [31]:
#EfficientNet Transfer Learning
base_model = tf.keras.applications.EfficientNetB0(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False   # Freeze feature extractor

inputs = layers.Input(shape=(224,224,3))

x = tf.keras.applications.efficientnet.preprocess_input(inputs)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x)

outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model_3 = models.Model(inputs, outputs)

model_3.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 247)            │       316,407 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,365,978 (16.65 MB)

 Trainable params: 316,407 (1.21 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [9]:
# compile
model_3.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
#Callbacks
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "best_model_3.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)


In [10]:
history_3 = model_3.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[checkpoint, early_stop]
)


Epoch 1/20
541/541 ━━━━━━━━━━━━━━━━━━━━ 0s 590ms/step - accuracy: 0.0499 - loss: 5.1353
Epoch 1: val_accuracy improved from -inf to 0.23168, saving model to best_model_3.keras
541/541 ━━━━━━━━━━━━━━━━━━━━ 398s 714ms/step - accuracy: 0.0500 - loss: 5.1344 - val_accuracy: 0.2317 - val_loss: 3.7764
Epoch 2/20
541/541 ━━━━━━━━━━━━━━━━━━━━ 0s 554ms/step - accuracy: 0.2235 - loss: 3.6691
Epoch 2: val_accuracy improved from 0.23168 to 0.33863, saving model to best_model_3.keras
541/541 ━━━━━━━━━━━━━━━━━━━━ 363s 670ms/step - accuracy: 0.2236 - loss: 3.6688 - val_accuracy: 0.3386 - val_loss: 3.1115
Epoch 3/20
541/541 ━━━━━━━━━━━━━━━━━━━━ 0s 552ms/step - accuracy: 0.3276 - loss: 3.0690
Epoch 3: val_accuracy improved from 0.33863 to 0.39547, saving model to best_model_3.keras
541/541 ━━━━━━━━━━━━━━━━━━━━ 362s 670ms/step - accuracy: 0.3276 - loss: 3.0688 - val_accuracy: 0.3955 - val_loss: 2.7537
Epoch 4/20
541/541 ━━━━━━━━━━━━━━━━━━━━ 0s 582ms/step - accuracy: 0.3914 - loss: 2.7096
Epoch 4: val_ac

In [11]:
loss, acc = model_3.evaluate(test_ds)
print("Test Accuracy =", acc)
print("Test Loss =", loss)


116/116 ━━━━━━━━━━━━━━━━━━━━ 72s 599ms/step - accuracy: 0.6251 - loss: 1.4803
Test Accuracy = 0.6149269938468933
Test Loss = 1.502914547920227


## Fine-Tuning

In [21]:
import tensorflow as tf

model_4 = tf.keras.models.load_model("best_model_3.keras")


In [22]:
model_4.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 247)            │       316,407 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,998,794 (19.07 MB)

 Trainable params: 316,407 (1.21 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

 Optimizer params: 632,816 (2.41 MB)

In [23]:
#Unfreeze Top Layers of EfficientNet
base_model = model_4.get_layer("efficientnetb0")
  # EfficientNet block

base_model.trainable = True

# Freeze bottom 70% and train only top 30%
for layer in base_model.layers[:200]:
    layer.trainable = False
#EfficientNetB0 has ~237 layers, adjust if needed

In [24]:
#Recompile With Smaller Learning Rate
model_4.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [25]:
#Callback
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "best_model_4.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)


In [32]:
#Train
history_4 = model_4.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[checkpoint, early_stop]
)



Epoch 1/30


541/541 ━━━━━━━━━━━━━━━━━━━━ 0s 817ms/step - accuracy: 0.8377 - loss: 0.5881
Epoch 1: val_accuracy improved from 0.81250 to 0.82085, saving model to best_model_4.keras
541/541 ━━━━━━━━━━━━━━━━━━━━ 513s 947ms/step - accuracy: 0.8377 - loss: 0.5880 - val_accuracy: 0.8209 - val_loss: 0.6422
Epoch 2/30
541/541 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8404 - loss: 0.5696
Epoch 2: val_accuracy improved from 0.82085 to 0.82247, saving model to best_model_4.keras
541/541 ━━━━━━━━━━━━━━━━━━━━ 796s 1s/step - accuracy: 0.8404 - loss: 0.5696 - val_accuracy: 0.8225 - val_loss: 0.6311
Epoch 3/30
541/541 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8429 - loss: 0.5552
Epoch 3: val_accuracy improved from 0.82247 to 0.82705, saving model to best_model_4.keras
541/541 ━━━━━━━━━━━━━━━━━━━━ 897s 2s/step - accuracy: 0.8430 - loss: 0.5552 - val_accuracy: 0.8270 - val_loss: 0.6218
Epoch 4/30
541/541 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8468 - loss: 0.5469
Epoch 4: val_accuracy improved from 0.

In [ ]:
#evaluvate
loss, acc = model_4.evaluate(test_ds)
print("Test Accuracy =", acc)
print("Test Loss =", loss)
